ED 408 Statistical Machine Learning II Lab

Vivek Kumar

IED/10008/22

December 1, 2025

In [ ]:

# Part 1 — Global configuration & constants
import os, time, math, random
import numpy as np
import pandas as pd


# Tickers (26-asset)
TICKERS = [
    # India Equities
    "NIFTYBEES.NS", "HDFCBANK.NS", "RELIANCE.NS", "INFY.NS", "TCS.NS", "ASIANPAINT.NS",
    # US ETFs / Equities
    "SPY", "QQQ", "IWM", "XLK", "XLF", "XLY", "SMH", "ARKK",
    # Global ex-US
    "EEM", "VGK", "EWJ",
    # Commodities
    "GLD", "SLV", "USO", "CPER",
    # Bonds
    "TLT", "IEF", "SHY",
    # Crypto
    "BTC-USD", "ETH-USD",
]

CRYPTO_TICKERS = ["BTC-USD", "ETH-USD"]
COMMODITY_TICKERS = ["GLD", "SLV", "USO", "CPER"]
BOND_TICKERS = ["TLT", "IEF", "SHY"]

# GA parameters
POP_SIZE = 80
N_GENERATIONS = 50
ENSEMBLE_RUNS = 2
TOURNAMENT_K = 3
ELITISM = 2
MUTATION_RATE = 0.2
MUTATION_STD = 0.03


USE_RANK_FITNESS = True
W_RET = 1.0
W_RISK = 1.0
W_TURN = 0.7
W_PCA = 0.5
CONSISTENCY_PENALTY = 0.8

# limits & caps
GROUP_LIMITS = {"crypto": 0.05, "commodities": 0.15, "bonds": 0.5}
MAX_ASSET_WEIGHT = 0.20

# Volatility veto & leverage defaults
VOL_SPIKE_K = 1.5
LEV_BAD = 0.8
LEV_GOOD = 1.2
LEV_MIN = 0.5
LEV_MAX = 1.5

# Blend & transaction defaults
BLEND_GOOD = 0.12
BLEND_BAD = 1.0
TC_BPS = 5  # transaction cost in bps
INITIAL_CAPITAL = 1_000_000
PLOT_DPI = 120
OUTDIR = "backtest_outputs"

os.makedirs(OUTDIR, exist_ok=True)

# Part 2 — Data loader & FX handling
import yfinance as yf

def download_price_data(tickers, start="2015-01-01", end=None, auto_adjust=True):
    if end is None:
        end = pd.Timestamp.today().strftime("%Y-%m-%d")

    raw = yf.download(tickers, start=start, end=end, progress=False, group_by='ticker')

    if isinstance(raw, pd.DataFrame) and "Close" in raw.columns:
        prices = raw["Close"].copy()
    else:
        # yfinance sometimes returns a multiindex
        try:
            # When grouped by ticker, raw[ticker]['Close']
            frames = []
            for t in tickers:
                try:
                    c = raw[t]["Close"].rename(t)
                    frames.append(c)
                except Exception:
                    pass
            prices = pd.concat(frames, axis=1)
        except Exception:
            # fallback
            prices = raw.copy()
    prices = prices.sort_index().ffill().bfill()
    return prices

def synthesize_fx(prices, local_bench="NIFTYBEES.NS", foreign_bench="SPY"):
    # synthetic USD->INR ratio from median SPY/NIFTYBEES ratio (heuristic)
    if local_bench in prices.columns and foreign_bench in prices.columns:
        ratio = prices[foreign_bench].median() / (prices[local_bench].median() + 1e-9)
        return ratio
    return None

def load_all_data(start="2015-01-01", end=None):
    print("Starting multi-ticker download...")
    prices = download_price_data(TICKERS, start=start, end=end)
    print("Downloaded tickers. Price shape:", prices.shape)

    # Ensure all tickers are present as columns; if some missing, warn and drop
    missing = [t for t in TICKERS if t not in prices.columns]
    if missing:
        print("Warning: missing tickers (dropped):", missing)
    prices = prices[[c for c in TICKERS if c in prices.columns]]

    usd_present = any(c.endswith("USD") or c in ["SPY","QQQ","IWM","ARKK","SMH"] for c in prices.columns)
    if usd_present and "NIFTYBEES.NS" in prices.columns:
        try:
            fx = yf.download("USDINR=X", start=start, end=end, progress=False)["Close"].ffill().bfill()
            if fx.isna().all():
                raise Exception("FX series empty")
            fx_median = fx.median()
            # convert USD tickers to INR (approx) by multiplying by FX
            for c in list(prices.columns):
                if (c.endswith("USD") or c in ["SPY","QQQ","IWM","ARKK","SMH"]) and c in prices.columns:
                    try:
                        prices[c] = prices[c].mul(fx.reindex(prices.index)).ffill().bfill()
                    except Exception:
                        pass
            print("FX conversion applied using USDINR=X")
        except Exception:
            # synthetic fx
            ratio = synthesize_fx(prices, local_bench="NIFTYBEES.NS", foreign_bench="SPY")
            if ratio is not None:
                for c in list(prices.columns):
                    if (c.endswith("USD") or c in ["SPY","QQQ","IWM","ARKK","SMH"]) and c in prices.columns:
                        prices[c] = prices[c] * ratio
                print("Synthetic FX created (median ratio).")
            else:
                print("FX conversion skipped (no synthetic available).")
    else:
        print("FX conversion not required.")
    pd.DataFrame({"shape_rows": [prices.shape[0]], "shape_cols":[prices.shape[1]]}).to_csv(os.path.join(OUTDIR,"post_conversion_diagnostics.csv"), index=False)
    print("Final prices shape:", prices.shape)
    return prices

# Part 3 — Feature engineering, HRP, PCA, enforce_group_limits
from sklearn.decomposition import PCA
import scipy.cluster.hierarchy as sch
import scipy.spatial.distance as ssd

def hrp_quasi_diag(link):
    link = np.array(link).astype(int)
    # total original items = number of leaves = link[-1, 3] if linkage matrix includes counts, else deduce
    n = link.shape[0] + 1
    def get_order(link):
        clusters = {i: [i] for i in range(n)}
        for i, row in enumerate(link):
            a, b = int(row[0]), int(row[1])
            clusters[n + i] = clusters.get(a, [a]) + clusters.get(b, [b])
        return clusters[ n + link.shape[0] - 1 ]
    order = get_order(link)
    return order

def get_cluster_var(cov, cluster):
    cov_ = cov[np.ix_(cluster, cluster)]
    w = np.ones(len(cluster)) / len(cluster)
    return float(w.T @ cov_ @ w)

def compute_hrp_weights(returns_window):
    # Returns numpy array of weights length = n_assets
    rets = returns_window.dropna(axis=1, how='all')
    cov = rets.cov().values
    corr = rets.corr().values
    

    dist = np.sqrt((1 - corr) / 2)
    # flatten lower triangle
    dist_vec = ssd.squareform(dist, checks=False)
    if np.isnan(dist_vec).any():
        # fallback to identity-like
        cov = np.nan_to_num(cov, nan=1e-6)
        dist_vec = ssd.squareform(np.sqrt((1 - np.nan_to_num(corr, nan=0)) / 2), checks=False)
    link = sch.linkage(dist_vec, method='single')
    # get order
    try:
        sort_ix = hrp_quasi_diag(link)
    except Exception:
        sort_ix = list(range(len(rets.columns)))
    # bisection weights
    n = len(rets.columns)
    w = np.ones(n)
    clusters = [list(range(n))]
    while clusters:
        new_clusters = []
        for cluster in clusters:
            if len(cluster) <= 1:
                continue
            split = int(len(cluster) / 2)
            c1 = cluster[:split]
            c2 = cluster[split:]
            var1 = get_cluster_var(cov, c1)
            var2 = get_cluster_var(cov, c2)
            alpha = 0.5 if (var1 + var2) == 0 else 1 - var1 / (var1 + var2)
            w[c1] = w[c1] * alpha
            w[c2] = w[c2] * (1 - alpha)
            new_clusters.extend([c1, c2])
        clusters = new_clusters
    # map back to original order
    # if rets.columns are not the original TICKERS subset, align by position
    final = np.zeros(len(rets.columns))
    final[:] = w / w.sum()
    # Expand to full TICKERS length: place weights in the columns present, zeros elsewhere
    full = np.zeros(len(TICKERS))
    for i, col in enumerate(rets.columns):
        if col in TICKERS:
            idx = TICKERS.index(col)
            full[idx] = final[i]
    if full.sum() <= 0:
        full = np.ones(len(TICKERS)) / len(TICKERS)
    else:
        full = full / full.sum()
    return full

def compute_pca_exposure(returns_window, n_components=1):
    try:
        R = returns_window.fillna(0.0).values
        if R.shape[0] < 2 or R.shape[1] < 1:
            return np.ones(len(TICKERS)) / len(TICKERS)
        pca = PCA(n_components=min(n_components, R.shape[1]))
        pca.fit(R)
        pc1 = np.abs(pca.components_[0])
        # map to tickers ordering of returns_window columns
        cols = list(returns_window.columns)
        vec = np.zeros(len(TICKERS))
        for i, c in enumerate(cols):
            if c in TICKERS:
                vec[TICKERS.index(c)] = pc1[i]
        if vec.sum() == 0:
            return np.ones(len(TICKERS)) / len(TICKERS)
        return vec / vec.sum()
    except Exception:
        return np.ones(len(TICKERS)) / len(TICKERS)

def compute_momentum(returns_window):
    return returns_window.mean() * 252

def compute_vol_forecast(returns_window):
    vol21 = returns_window.rolling(21).std().iloc[-1]
    vol63 = returns_window.rolling(63).std().iloc[-1]
    # map to TICKERS
    vec21 = np.zeros(len(TICKERS)); vec63 = np.zeros(len(TICKERS))
    for i, t in enumerate(TICKERS):
        if t in vol21.index:
            vec21[i] = vol21.loc[t]
        if t in vol63.index:
            vec63[i] = vol63.loc[t]
    return vec21, vec63

def enforce_group_limits(weights):
    # accepts numpy array or pd.Series, returns numpy array normalized
    w = np.array(weights, dtype=float).copy()
    if w.size != len(TICKERS):
        # try align if Series with index
        try:
            import pandas as pd
            if isinstance(weights, pd.Series):
                tmp = np.zeros(len(TICKERS))
                for i, t in enumerate(TICKERS):
                    if t in weights.index:
                        tmp[i] = weights.loc[t]
                w = tmp
        except Exception:
            # fallback pad/truncate
            if w.size < len(TICKERS):
                new = np.zeros(len(TICKERS))
                new[:w.size] = w
                w = new
            else:
                w = w[:len(TICKERS)]
    # avoid zeros
    if w.sum() <= 0:
        w = np.ones(len(TICKERS)) / len(TICKERS)
    # group caps
    name_list = TICKERS
    def _scale_group(group_list, cap):
        idx = [name_list.index(t) for t in group_list if t in name_list]
        if not idx:
            return
        s = w[idx].sum()
        if s > cap and s > 0:
            w[idx] = w[idx] * (cap / s)
    try:
        _scale_group(CRYPTO_TICKERS, GROUP_LIMITS.get("crypto", 0.05))
        _scale_group(COMMODITY_TICKERS, GROUP_LIMITS.get("commodities", 0.15))
        _scale_group(BOND_TICKERS, GROUP_LIMITS.get("bonds", 0.5))
    except Exception:
        pass
    # per-asset cap
    w = np.minimum(w, MAX_ASSET_WEIGHT)
    # renormalize
    if w.sum() <= 0:
        w = np.ones(len(TICKERS)) / len(TICKERS)
    else:
        w = w / w.sum()
    return w

# Part 4 — Genetic Algorithm (rank-based fitness, penalties, ensemble)
def compute_cvar(series, alpha=0.05):
    arr = np.asarray(series)
    if arr.size == 0:
        return 0.0
    sorted_rets = np.sort(arr)
    k = max(1, int(alpha * len(sorted_rets)))
    return float(-np.mean(sorted_rets[:k]))

def evaluate_population(pop_weights, features, prev_weights, prices_window, prices_next):
    # robust evaluation function
    import pandas as pd
    returns_w = features.get("returns_window")
    cov = features.get("cov")
    hrp_w = features.get("hrp_weights")
    # returns matrix
    if isinstance(returns_w, pd.DataFrame):
        returns_vals = returns_w.values
        columns = list(returns_w.columns)
    else:
        returns_vals = np.zeros((1, len(TICKERS)))
        columns = TICKERS
    n_assets = len(TICKERS)
    # cov fallback
    cov = np.asarray(cov) if cov is not None else np.eye(n_assets) * 1e-6
    if cov.shape != (n_assets, n_assets):
        try:
            cov = np.cov(returns_vals, rowvar=False)
        except Exception:
            cov = np.eye(n_assets) * 1e-6
    # forward return vector
    if prices_next is not None and prices_window is not None:
        try:
            p_last = prices_window.iloc[-1].values
            fwd_ret_vec = (prices_next.values / p_last) - 1.0
        except Exception:
            fwd_ret_vec = returns_vals[-1] if returns_vals.ndim == 2 else np.zeros(n_assets)
    else:
        fwd_ret_vec = returns_vals[-1] if returns_vals.ndim == 2 else np.zeros(n_assets)
    fwd_ret_vec = np.resize(np.asarray(fwd_ret_vec).flatten(), n_assets)
    results = []
    prev_w = np.asarray(prev_weights, dtype=float).flatten()
    prev_w = prev_w if prev_w.size == n_assets else np.ones(n_assets)/n_assets
    for w in pop_weights:
        w = np.asarray(w, dtype=float).flatten()
        if w.size != n_assets:
            w = np.resize(w, n_assets)
        w = np.clip(w, 0.0, MAX_ASSET_WEIGHT)
        if w.sum() <= 0:
            w = np.ones(n_assets) / n_assets
        else:
            w = w / w.sum()
        try:
            w = enforce_group_limits(w)
        except Exception:
            pass
        # metrics
        ret_raw = float(np.dot(w, fwd_ret_vec))
        port_var = float(np.dot(w, np.dot(cov, w)))
        port_std = math.sqrt(max(port_var, 0.0))
        try:
            port_series = np.dot(returns_vals, w)
            cvar = compute_cvar(port_series, alpha=0.05)
        except Exception:
            cvar = 0.0
            port_series = np.zeros(max(1, returns_vals.shape[0]))
        pca_vec = compute_pca_exposure(pd.DataFrame(returns_vals, columns=columns))
        pca_exp = float(np.sum(np.abs(pca_vec * w))) if hasattr(pca_vec, "__len__") else float(pca_vec)
        turnover = float(np.sum(np.abs(w - prev_w)))
        consistency = float(np.sum((w - prev_w)**2))
        cum = np.cumsum(port_series)
        dd = float((cum - np.maximum.accumulate(cum)).min()) if cum.size>0 else 0.0
        dd_pen = -dd if dd < 0 else 0.0
        results.append({
            "weights": w,
            "ret": ret_raw,
            "std": port_std,
            "cvar": cvar,
            "pca_exp": pca_exp,
            "turnover": turnover,
            "consistency": consistency,
            "dd_pen": dd_pen
        })
    # fitness calculation
    try:
        if USE_RANK_FITNESS:
            rets = np.array([r["ret"] for r in results])
            risks = np.array([r["cvar"] for r in results])
            turns = np.array([r["turnover"] for r in results])
            pcas = np.array([r["pca_exp"] for r in results])
            N = len(results)
            ret_rank = (N - np.argsort(np.argsort(rets))) / float(N)
            risk_rank = (np.argsort(np.argsort(risks))) / float(N)
            turn_rank = (np.argsort(np.argsort(turns))) / float(N)
            pca_rank = (np.argsort(np.argsort(pcas))) / float(N)
            for i, r in enumerate(results):
                fitness = (W_RET * ret_rank[i] - W_RISK * risk_rank[i] - W_TURN * turn_rank[i] - W_PCA * pca_rank[i])
                fitness -= CONSISTENCY_PENALTY * r["consistency"]
                r["fitness"] = float(fitness)
        else:
            def z(x):
                x = np.asarray(x, dtype=float)
                return (x - x.mean()) / (x.std() + 1e-9)
            rets_z = z([r["ret"] for r in results])
            risks_z = z([r["cvar"] for r in results])
            turns_z = z([r["turnover"] for r in results])
            pcas_z = z([r["pca_exp"] for r in results])
            for i, r in enumerate(results):
                fitness = W_RET * rets_z[i] - W_RISK * risks_z[i] - W_TURN * turns_z[i] - W_PCA * pcas_z[i]
                fitness -= CONSISTENCY_PENALTY * r["consistency"]
                r["fitness"] = float(fitness)
    except Exception:
        for r in results:
            r["fitness"] = float(r["ret"] - r["cvar"])
    return results

def tournament_select(pop_results, k=TOURNAMENT_K):
    selected = random.sample(pop_results, min(k, len(pop_results)))
    selected.sort(key=lambda x: x["fitness"], reverse=True)
    return selected[0]["weights"]

def crossover(w1, w2):
    if random.random() < 0.5:
        pt = random.randint(1, len(w1)-1)
        child = np.concatenate([w1[:pt], w2[pt:]])
    else:
        alpha = random.random()
        child = alpha * w1 + (1-alpha) * w2
    return child

def mutate(w):
    if random.random() < MUTATION_RATE:
        noise = np.random.normal(0, MUTATION_STD, size=len(w))
        w_new = w + noise
        w_new = np.clip(w_new, 0.0, MAX_ASSET_WEIGHT)
        if w_new.sum() <= 0:
            w_new = np.ones_like(w_new) / len(w_new)
        else:
            w_new = w_new / w_new.sum()
        return w_new
    return w

def run_ga_single(features, prev_weights, prices_window, prices_next):
    n_assets = len(TICKERS)
    # warm-start population
    pop = []
    # seed with prev_weights variants
    for _ in range(max(2, int(POP_SIZE*0.1))):
        noise = np.random.normal(0, 0.02, n_assets)
        w = np.clip(prev_weights + noise, 0, MAX_ASSET_WEIGHT)
        w = w / (w.sum() + 1e-12)
        pop.append(w)
    # seed with HRP variants
    for _ in range(max(1, int(POP_SIZE*0.05))):
        try:
            w = enforce_group_limits(features["hrp_weights"] + np.random.normal(0, 0.01, n_assets))
            pop.append(w)
        except Exception:
            pass
    # random fill
    while len(pop) < POP_SIZE:
        r = np.random.rand(n_assets)
        r = np.clip(r, 0, 1)
        r = r / r.sum()
        pop.append(r)
    best_w = pop[0]
    best_f = -1e9
    for gen in range(N_GENERATIONS):
        pop_results = evaluate_population(pop, features, prev_weights, prices_window, prices_next)
        pop_results.sort(key=lambda x: x["fitness"], reverse=True)
        if pop_results[0]["fitness"] > best_f:
            best_f = pop_results[0]["fitness"]
            best_w = pop_results[0]["weights"].copy()
        # new pop
        new_pop = []
        for i in range(ELITISM):
            new_pop.append(pop_results[i]["weights"])
        while len(new_pop) < POP_SIZE:
            p1 = tournament_select(pop_results)
            p2 = tournament_select(pop_results)
            child = crossover(p1, p2)
            child = mutate(child)
            child = enforce_group_limits(child)
            new_pop.append(child)
        pop = new_pop
    return enforce_group_limits(best_w)

def run_ga_ensemble(features, prev_weights, prices_window, prices_next):
    ws = []
    for _ in range(max(1, ENSEMBLE_RUNS)):
        try:
            w = run_ga_single(features, prev_weights, prices_window, prices_next)
            ws.append(w)
        except Exception as e:
            # fail quietly
            pass
    if len(ws) == 0:
        return enforce_group_limits(features.get("hrp_weights", np.ones(len(TICKERS))/len(TICKERS)))
    mean_w = np.mean(np.vstack(ws), axis=0)
    return enforce_group_limits(mean_w)

# Part 5 — Walk-forward backtest engine (monthly rebalance default)
def compute_hrm_features(prices_window):
    rets = prices_window.pct_change().dropna(how='all')
    cov = rets.cov()
    momentum = compute_momentum(rets)
    vol_short = rets.rolling(21).std().iloc[-1]
    vol_long = rets.rolling(63).std().iloc[-1]
    hrp_w = compute_hrp_weights(rets)
    return {
        "returns_window": rets,
        "cov": cov.values,
        "momentum": momentum,
        "vol_short": vol_short,
        "vol_long": vol_long,
        "hrp_weights": hrp_w
    }

def detect_hmm_regime(returns_window):
    # lightweight HMM placeholder: use vol ratio heuristic inside HMM call
    # returns True if "bad" regime detected
    vol21 = returns_window.rolling(21).std().mean(axis=1).iloc[-1] if hasattr(returns_window, "rolling") else np.nan
    vol63 = returns_window.rolling(63).std().mean(axis=1).iloc[-1] if hasattr(returns_window, "rolling") else np.nan
    try:
        return vol21 > VOL_SPIKE_K * vol63
    except Exception:
        return False

def backtest_pipeline(prices, rebalance_freq=21, train_window=252*2, tc_bps=TC_BPS, use_hmm=True, use_vol_spike=True):
    # defensive copy
    prices = prices.copy().ffill().bfill()
    rets = prices.pct_change().fillna(0.0)
    nav = INITIAL_CAPITAL
    prev_weights = np.ones(len(TICKERS)) / len(TICKERS)
    weight_history = []
    nav_history = []
    lev_history = []
    meta = []
    dates = list(prices.index)
    # iterate
    idx_start = train_window
    idx_end = len(prices) - 1
    for t in range(idx_start, idx_end, rebalance_freq):
        date = prices.index[t]
        date_next = prices.index[t+1]
        prices_win = prices.iloc[t-train_window:t+1]  # include current
        rets_win = prices_win.pct_change().dropna(how='all')
        features = compute_hrm_features(prices_win)
        # regime detection
        hmm_bad = False
        if use_hmm:
            hmm_bad = detect_hmm_regime(rets_win)
        vol_spike = False
        try:
            vs = features["vol_short"].mean() if isinstance(features["vol_short"], (pd.Series, np.ndarray)) else np.nan
            vl = features["vol_long"].mean() if isinstance(features["vol_long"], (pd.Series, np.ndarray)) else np.nan
            if use_vol_spike and not np.isnan(vs) and not np.isnan(vl) and vs > VOL_SPIKE_K * vl:
                vol_spike = True
        except Exception:
            vol_spike = False
        bad_regime = hmm_bad or vol_spike
        # GA or HRP
        if bad_regime:
            ga_w = features["hrp_weights"].copy()
        else:
            ga_w = run_ga_ensemble(features, prev_weights, prices_win, prices.iloc[t+1])
        # blend
        blend = BLEND_BAD if bad_regime else BLEND_GOOD
        hrp_w = features["hrp_weights"]
        final_w = blend * hrp_w + (1 - blend) * ga_w
        final_w = np.clip(final_w, 0.0, MAX_ASSET_WEIGHT)
        if final_w.sum() <= 0:
            final_w = np.ones(len(TICKERS)) / len(TICKERS)
        final_w = final_w / final_w.sum()
        final_w = enforce_group_limits(final_w)
        # leverage
        if bad_regime:
            lev = LEV_BAD
        else:
            ratio = 1.0
            try:
                vs = np.nanmean(features["vol_short"]) if hasattr(features["vol_short"], "__len__") else float(features["vol_short"])
                vl = np.nanmean(features["vol_long"]) if hasattr(features["vol_long"], "__len__") else float(features["vol_long"])
                ratio = float(vs / (vl + 1e-9))
            except Exception:
                ratio = 1.0
            lev = float(max(LEV_BAD, min(LEV_GOOD, ratio)))
            lev = max(LEV_MIN, min(LEV_MAX, lev))
        # turnover & cost
        turnover = float(np.sum(np.abs(final_w - prev_weights)))
        trade_cost = turnover * (tc_bps / 10000.0) * nav
        # apply next-day returns
        r_vec = (prices.iloc[t+1] / prices.iloc[t] - 1.0).fillna(0.0).values
        port_ret = lev * float(np.dot(final_w, r_vec))
        nav_before = nav
        nav = nav * (1 + port_ret) - trade_cost
        nav_history.append((prices.index[t+1], nav))
        weight_history.append((prices.index[t+1], final_w.copy()))
        lev_history.append((prices.index[t+1], lev))
        meta.append({
            "date": prices.index[t+1],
            "nav_before": nav_before,
            "nav_after_trade": nav,
            "turnover": turnover,
            "trade_cost": trade_cost,
            "blend_factor": blend,
            "bad_veto": bool(bad_regime),
            "lev": lev
        })
        prev_weights = final_w.copy()
    # convert to DataFrames
    nav_df = pd.DataFrame(nav_history, columns=["date", "NAV"]).set_index("date")
    weights_df = pd.DataFrame([ [d[0]] + list(d[1]) for d in weight_history ], columns=["date"] + TICKERS).set_index("date")
    lev_df = pd.DataFrame(lev_history, columns=["date","leverage"]).set_index("date")
    meta_df = pd.DataFrame(meta)
    # benchmark NAV: pick BENCH_TICKER if exists else first available column
    BENCH_TICKER = globals().get("BENCH_TICKER", None)
    if BENCH_TICKER is None or BENCH_TICKER not in prices.columns:
        # choose sensible
        for cand in ["NIFTYBEES.NS","^NSEI","SPY","QQQ"]:
            if cand in prices.columns:
                BENCH_TICKER = cand
                break
    bench_nav = None
    try:
        bench_rets = prices[BENCH_TICKER].pct_change().reindex(nav_df.index).fillna(0.0)
        bench_nav = (1 + bench_rets).cumprod() * INITIAL_CAPITAL
    except Exception:
        pass
    # annual returns
    ann = nav_df.copy()
    ann["Year"] = ann.index.year
    try:
        annual = ann.groupby("Year").apply(lambda x: x["NAV"].iloc[-1] / x["NAV"].iloc[0] - 1)
    except Exception:
        annual = pd.Series(dtype=float)
    bench_annual = pd.Series(dtype=float)
    if bench_nav is not None:
        try:
            bdf = bench_nav.to_frame(name="BENCH")
            bdf["Year"] = bdf.index.year
            bench_annual = bdf.groupby("Year").apply(lambda x: x.iloc[-1,0] / x.iloc[0,0] - 1)
        except Exception:
            bench_annual = pd.Series(dtype=float)
    annual_table = pd.DataFrame({"Strategy": annual, "Benchmark": bench_annual}).fillna(0.0)
    # summary stats
    total_return = float(nav_df["NAV"].iloc[-1] / nav_df["NAV"].iloc[0] - 1) if not nav_df.empty else 0.0
    yrs = (nav_df.index[-1] - nav_df.index[0]).days / 365.0 if len(nav_df.index)>1 else 1.0
    cagr = (1 + total_return)**(1.0/max(yrs,1e-9)) - 1 if yrs>0 else 0.0
    ann_vol = float(nav_df["NAV"].pct_change().std() * math.sqrt(252)) if not nav_df.empty else 0.0
    sharpe = float((nav_df["NAV"].pct_change().mean() * 252) / (ann_vol + 1e-9)) if ann_vol>0 else 0.0
    dd = float((nav_df["NAV"] / nav_df["NAV"].cummax() - 1).min()) if not nav_df.empty else 0.0
    stats = {"total_return": total_return, "CAGR": cagr, "ann_vol": ann_vol, "sharpe": sharpe, "maxdd": dd}
    # save outputs
    os.makedirs(OUTDIR, exist_ok=True)
    nav_df.to_csv(os.path.join(OUTDIR,"nav.csv"))
    weights_df.to_csv(os.path.join(OUTDIR,"weights.csv"))
    lev_df.to_csv(os.path.join(OUTDIR,"leverage.csv"))
    meta_df.to_csv(os.path.join(OUTDIR,"rebalance_meta.csv"), index=False)
    annual_table.to_csv(os.path.join(OUTDIR,"annual_returns.csv"))
    pd.DataFrame([stats]).to_csv(os.path.join(OUTDIR,"summary_stats.csv"), index=False)
    print("Backtest complete. Outputs written to", OUTDIR)
    return {"nav": nav_df, "weights": weights_df, "leverage": lev_df, "meta": meta_df, "annual": annual_table, "stats": stats, "bench_nav": bench_nav}

def run_all_and_report(prices):
    out = backtest_pipeline(prices)
    print("Summary stats:")
    print(out["stats"])
    print("Annual returns table (saved):")
    print(out["annual"].head(20))
    return out, out["stats"], out["annual"]

# Part 6 — Reporting, plotting & ablations
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import statsmodels.api as sm

def load_backtest_outputs(outdir=OUTDIR):
    nav_df = pd.read_csv(os.path.join(outdir,"nav.csv"), index_col=0, parse_dates=True)
    if isinstance(nav_df, pd.Series):
        nav_df = nav_df.to_frame(name=nav_df.name if nav_df.name else "NAV")
    if "NAV" not in nav_df.columns:
        if nav_df.shape[1] == 1:
            nav_df.columns = ["NAV"]
        else:
            # try find numeric column
            for c in nav_df.columns:
                if pd.api.types.is_numeric_dtype(nav_df[c]):
                    nav_df = nav_df[[c]].rename(columns={c:"NAV"})
                    break
    weights = pd.read_csv(os.path.join(outdir,"weights.csv"), index_col=0, parse_dates=True)
    lev = pd.read_csv(os.path.join(outdir,"leverage.csv"), index_col=0, parse_dates=True)
    if isinstance(lev, pd.DataFrame) and lev.shape[1]==1:
        lev = lev.iloc[:,0]
    meta = pd.read_csv(os.path.join(outdir,"rebalance_meta.csv"), parse_dates=["date"])
    bench = None
    bench_path = os.path.join(outdir,"bench_nav.csv")
    if os.path.exists(bench_path):
        bench = pd.read_csv(bench_path, index_col=0, parse_dates=True)
        if isinstance(bench, pd.DataFrame) and bench.shape[1]==1:
            bench = bench.iloc[:,0]
    return nav_df, weights, lev, meta, bench

def plot_allocation_history(weights_df, leverage_series=None, top_n=12, filename="allocation_history.png"):
    w = weights_df.fillna(0.0).copy()
    w.index = pd.to_datetime(w.index)
    means = w.mean().sort_values(ascending=False)
    top = list(means.head(top_n).index)
    others = [c for c in w.columns if c not in top]
    if others:
        w["Other"] = w[others].sum(axis=1)
        w = w[top + ["Other"]]
    else:
        w = w[top]
    row_sum = w.sum(axis=1).replace(0,1.0)
    w_norm = w.div(row_sum, axis=0)
    fig, ax = plt.subplots(figsize=(14,5))
    w_norm.plot.area(ax=ax, linewidth=0)
    ax.set_title("Allocation History (stacked weights)")
    ax.set_ylabel("Unlevered Weight")
    ax.set_ylim(0,1.0)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
    if leverage_series is not None:
        lev = leverage_series.reindex(w_norm.index).fillna(method='ffill').fillna(1.0)
        ax2 = ax.twinx()
        ax2.plot(lev.index, lev.values, color='k', linewidth=1.2, label='Leverage')
        ax2.set_ylabel("Leverage")
    out = os.path.join(OUTDIR, filename)
    plt.tight_layout()
    fig.savefig(out, dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)
    print("Saved allocation history ->", out)
    return w_norm

def plot_leverage_vs_vol(leverage_series, nav_series, benchmark_series=None, vol_window=21, filename="leverage_vs_vol.png"):
    lev = leverage_series.reindex(nav_series.index).fillna(method='ffill').fillna(1.0)
    strat_rets = nav_series.pct_change().fillna(0.0)
    vol_strat = strat_rets.rolling(vol_window).std() * math.sqrt(252)
    vol_plot = vol_strat if benchmark_series is None else benchmark_series.pct_change().reindex(nav_series.index).rolling(vol_window).std() * math.sqrt(252)
    idx = lev.index.intersection(vol_plot.index)
    lev_a = lev.reindex(idx)
    vol_a = vol_plot.reindex(idx)
    fig, ax1 = plt.subplots(figsize=(12,4))
    ax1.plot(lev_a.index, lev_a.values, label="Leverage", color='C0', linewidth=1.5)
    ax1.set_ylabel("Leverage", color='C0')
    ax2 = ax1.twinx()
    ax2.plot(vol_a.index, vol_a.values, label="Volatility", color='C1', linewidth=1.2)
    ax2.set_ylabel("Annualized Vol", color='C1')
    ax1.set_title(f"Leverage vs Volatility (window={vol_window}d)")
    fig.tight_layout()
    out = os.path.join(OUTDIR, filename)
    fig.savefig(out, dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)
    print("Saved leverage vs vol ->", out)
    return lev_a, vol_a

def plot_monthly_heatmap(nav_series, filename="monthly_returns_heatmap.png"):
    nav = nav_series.dropna().copy()
    rets = nav.pct_change().fillna(0.0)
    df = rets.groupby([rets.index.year, rets.index.month]).apply(lambda x: (1 + x).prod() - 1)
    df = df.unstack(level=1).reindex(columns=range(1,13))
    df.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4*len(df.index))))
    im = ax.imshow(df.values, aspect='auto', cmap="RdYlGn", vmin=-0.6, vmax=0.6)
    ax.set_yticks(np.arange(len(df.index))); ax.set_yticklabels(df.index)
    ax.set_xticks(np.arange(len(df.columns))); ax.set_xticklabels(df.columns)
    ax.set_title("Monthly Returns Heatmap (Strategy)")
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.iat[i,j]
            if not np.isnan(val):
                ax.text(j, i, f"{val*100:0.1f}%", ha='center', va='center', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    out = os.path.join(OUTDIR, filename)
    fig.tight_layout()
    fig.savefig(out, dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)
    print("Saved monthly heatmap ->", out)
    return df

def regime_average_weights(weights_df, nav_series, vol_window=21, filename="regime_avg_weights.csv"):
    w = weights_df.copy().fillna(0.0)
    rets = nav_series.pct_change().fillna(0.0)
    vol = rets.rolling(vol_window).std() * math.sqrt(252)
    thresh = vol.median()
    low_idx = vol[vol <= thresh].index
    high_idx = vol[vol > thresh].index
    low_avg = w.loc[w.index.intersection(low_idx)].mean() if len(low_idx)>0 else pd.Series(0.0, index=w.columns)
    high_avg = w.loc[w.index.intersection(high_idx)].mean() if len(high_idx)>0 else pd.Series(0.0, index=w.columns)
    df = pd.concat([low_avg, high_avg], axis=1)
    df.columns = ["AvgWeight_LowVol", "AvgWeight_HighVol"]
    df = df.fillna(0.0).sort_values("AvgWeight_LowVol", ascending=False)
    df.to_csv(os.path.join(OUTDIR, filename))
    print("Saved regime-average weights ->", os.path.join(OUTDIR, filename))
    return df

def rolling_beta(strategy_nav, benchmark_nav, window=126, filename="rolling_beta.png"):
    s = strategy_nav.pct_change().dropna()
    b = benchmark_nav.reindex(s.index).pct_change().dropna()
    betas = pd.Series(index=s.index, dtype=float)
    for i in range(window, len(s)):
        y = s.iloc[i-window:i]
        x = b.iloc[i-window:i]
        if x.std() == 0:
            betas.iloc[i] = np.nan
            continue
        X = sm.add_constant(x.values)
        model = sm.OLS(y.values, X).fit()
        betas.iloc[i] = float(model.params[1])
    betas = betas.fillna(method='ffill').fillna(method='bfill')
    fig, ax = plt.subplots(figsize=(12,3))
    ax.plot(betas.index, betas.values, linewidth=1)
    ax.set_title(f"Rolling Beta (window={window} days)")
    ax.set_ylabel("Beta")
    ax.axhline(1.0, color='k', linestyle='--', alpha=0.6)
    out = os.path.join(OUTDIR, filename)
    fig.tight_layout()
    fig.savefig(out, dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)
    print("Saved rolling beta ->", out)
    return betas

def bootstrap_annual_diff(annual_table_file=os.path.join(OUTDIR,"annual_returns.csv"), nboots=5000, seed=42):
    tbl = pd.read_csv(annual_table_file, index_col=0)
    strat = tbl["Strategy"].values
    bench = tbl["Benchmark"].values if "Benchmark" in tbl.columns else np.zeros_like(strat)
    obs_diff = np.mean(strat - bench)
    rng = np.random.default_rng(seed)
    boot_diffs = np.empty(nboots)
    n = len(strat)
    for i in range(nboots):
        idx = rng.integers(0, n, n)
        boot_diffs[i] = np.mean(strat[idx] - bench[idx])
    pval = np.mean(np.abs(boot_diffs) >= abs(obs_diff))
    ci = np.percentile(boot_diffs, [2.5, 97.5])
    out = {"obs_diff": float(obs_diff), "ci_low": float(ci[0]), "ci_high": float(ci[1]), "pval": float(pval)}
    pd.DataFrame([out]).to_csv(os.path.join(OUTDIR, "bootstrap_annual_diff.csv"), index=False)
    print("Bootstrap test complete ->", out)
    return out, boot_diffs

def run_ablation_experiments(prices, save_prefix="ablation"):
    backup_ensemble = ENSEMBLE_RUNS
    results = {}
    try:
        # HRP-only
        globals()['ENSEMBLE_RUNS'] = 0
        out_hrp = backtest_pipeline(prices, use_hmm=False, use_vol_spike=True)
        pd.DataFrame([out_hrp["stats"]]).to_csv(os.path.join(OUTDIR, f"{save_prefix}_hrp_stats.csv"), index=False)
        results["HRP_only"] = out_hrp["stats"]
        # 1/N
        tickers = [c for c in prices.columns]
        one_n_w = np.ones(len(tickers))/len(tickers)
        one_n_nav = (1 + prices[tickers].pct_change().fillna(0.0).dot(one_n_w)).cumprod() * INITIAL_CAPITAL
        stats_1n = {"total_return": float(one_n_nav.iloc[-1]/one_n_nav.iloc[0]-1)}
        pd.DataFrame([stats_1n]).to_csv(os.path.join(OUTDIR, f"{save_prefix}_1n_stats.csv"), index=False)
        results["1N"] = stats_1n
        # 60/40 (approx)
        equity_tickers = [t for t in prices.columns if t in ["SPY","QQQ","IWM","EEM","VGK","EWJ","ARKK","SMH","XLY","XLK","XLF"]]
        bond_tickers = [t for t in prices.columns if t in BOND_TICKERS]
        if equity_tickers and bond_tickers:
            eq_w = np.ones(len(equity_tickers))/len(equity_tickers)
            bd_w = np.ones(len(bond_tickers))/len(bond_tickers)
            daily_eq = prices[equity_tickers].pct_change().fillna(0.0).dot(eq_w)
            daily_bd = prices[bond_tickers].pct_change().fillna(0.0).dot(bd_w)
            combined_nav = (0.6 * (1+daily_eq).cumprod() + 0.4 * (1+daily_bd).cumprod())
            combined_nav = combined_nav/combined_nav.iloc[0]*INITIAL_CAPITAL
            stats_6040 = {"total_return": float(combined_nav.iloc[-1]/combined_nav.iloc[0]-1)}
            pd.DataFrame([stats_6040]).to_csv(os.path.join(OUTDIR, f"{save_prefix}_6040_stats.csv"), index=False)
            results["60/40"] = stats_6040
        # BTC BH
        if "BTC-USD" in prices.columns:
            btc_nav = (1 + prices["BTC-USD"].pct_change().fillna(0.0)).cumprod()*INITIAL_CAPITAL
            stats_btc = {"total_return": float(btc_nav.iloc[-1]/btc_nav.iloc[0]-1)}
            pd.DataFrame([stats_btc]).to_csv(os.path.join(OUTDIR, f"{save_prefix}_btc_stats.csv"), index=False)
            results["BTC"] = stats_btc
    finally:
        globals()['ENSEMBLE_RUNS'] = backup_ensemble
    print("Ablation experiments complete. Results saved to", OUTDIR)
    return results

def plot_nav_comparison(strategy_nav, benchmark_nav=None, btc_nav=None, gold_nav=None, filename="nav_comparison.png"):
    fig, ax = plt.subplots(figsize=(12,4))
    s = strategy_nav/strategy_nav.iloc[0]
    ax.plot(s.index, s.values, label="Strategy")
    if benchmark_nav is not None:
        b = benchmark_nav/benchmark_nav.iloc[0]
        ax.plot(b.index, b.values, label="Benchmark")
    if btc_nav is not None:
        bx = btc_nav/btc_nav.iloc[0]
        ax.plot(bx.index, bx.values, label="BTC")
    if gold_nav is not None:
        g = gold_nav/gold_nav.iloc[0]
        ax.plot(g.index, g.values, label="Gold")
    ax.set_title("NAV Comparison (normalized)")
    ax.legend()
    out = os.path.join(OUTDIR, filename)
    fig.tight_layout()
    fig.savefig(out, dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)
    print("Saved NAV comparison ->", out)
    return out

def export_summary_tables(nav, weights, leverage, meta, annual_table, outdir=OUTDIR):
    nav.to_csv(os.path.join(outdir, "nav.csv"))
    weights.to_csv(os.path.join(outdir, "weights.csv"))
    leverage.to_frame(name="leverage").to_csv(os.path.join(outdir, "leverage.csv"))
    meta.to_csv(os.path.join(outdir, "rebalance_meta.csv"), index=False)
    annual_table.to_csv(os.path.join(outdir, "annual_returns_table.csv"))
    print("Exported summary CSVs to", outdir)

def run_reporting_and_ablation(prices):
    nav, weights, lev, meta, bench_series = load_backtest_outputs(OUTDIR)
    export_summary_tables(nav, weights, lev, meta, pd.read_csv(os.path.join(OUTDIR,"annual_returns.csv"), index_col=0))
    plot_allocation_history(weights, lev, top_n=12)
    plot_leverage_vs_vol(lev, nav["NAV"], benchmark_series=bench_series)
    plot_monthly_heatmap(nav["NAV"])
    regime_average_weights(weights, nav["NAV"])
    if bench_series is not None:
        rolling_beta(nav["NAV"], bench_series)
    run_ablation_experiments(prices)
    try:
        bootstrap_annual_diff(os.path.join(OUTDIR,"annual_returns.csv"))
    except Exception:
        pass
    btc_nav = (1 + prices["BTC-USD"].pct_change().fillna(0.0)).cumprod()*INITIAL_CAPITAL if "BTC-USD" in prices.columns else None
    gold_nav = (1 + prices["GLD"].pct_change().fillna(0.0)).cumprod()*INITIAL_CAPITAL if "GLD" in prices.columns else None
    plot_nav_comparison(nav["NAV"], benchmark_nav=bench_series, btc_nav=btc_nav, gold_nav=gold_nav)
    print("Reporting complete. Files in", OUTDIR)
    return True

# Part 7 — Appendix: diagnostics & utilities
import hashlib, json, statsmodels.api as sm

def bootstrap_mean_diff(series_a, series_b, n=5000, seed=42):
    rng = np.random.default_rng(seed)
    A = np.asarray(series_a); B = np.asarray(series_b)
    mask = ~np.isnan(A) & ~np.isnan(B)
    A = A[mask]; B = B[mask]
    obs = A.mean() - B.mean() if len(A)>0 else 0.0
    diffs = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(A), len(A))
        diffs[i] = A[idx].mean() - B[idx].mean()
    ci = np.percentile(diffs, [2.5,97.5])
    p = np.mean(np.abs(diffs) >= abs(obs))
    return {"obs_diff": float(obs), "ci_low": float(ci[0]), "ci_high": float(ci[1]), "pval": float(p)}

def compute_rolling_beta(strategy_nav, bench_nav, window=126):
    sret = strategy_nav.pct_change().fillna(0.0)
    bret = bench_nav.pct_change().fillna(0.0)
    betas = []
    idxs = []
    for i in range(window, len(sret)):
        y = sret.iloc[i-window:i]; x = bret.iloc[i-window:i]
        cov = np.cov(y, x)[0,1]; var = np.var(x)
        betas.append(cov/(var+1e-9)); idxs.append(strategy_nav.index[i])
    return pd.Series(betas, index=idxs, name="RollingBeta")

def summarize_weights(weights_df, top_n=10):
    mean_w = weights_df.mean().sort_values(ascending=False)
    return mean_w.head(top_n), mean_w.tail(5)

def record_metadata(params_dict, outdir=OUTDIR):
    data = {"timestamp": time.ctime(), "params": params_dict}
    with open(os.path.join(outdir,"run_metadata.json"), "w") as f:
        json.dump(data, f, indent=2)
    print("Saved metadata ->", os.path.join(outdir,"run_metadata.json"))

def check_parts_loaded():
    checks = {
        "Part1 config": all(k in globals() for k in ["TICKERS","GROUP_LIMITS","POP_SIZE"]),
        "Part2 loader": "load_all_data" in globals(),
        "Part3 HRP/PCA": "compute_hrp_weights" in globals(),
        "Part4 GA": "run_ga_ensemble" in globals(),
        "Part5 backtest": "backtest_pipeline" in globals(),
        "Part6 reporting": "run_reporting_and_ablation" in globals()
    }
    print("======== Part Loading Status ========")
    for k,v in checks.items():
        print(f"{k:20s}: {'OK' if v else 'MISSING'}")
    print("=====================================")



Pipeline loaded. Run the following to execute:
prices = load_all_data()
out = backtest_pipeline(prices, use_hmm=False, use_vol_spike=True)
run_reporting_and_ablation(prices)


In [ ]:
prices = load_all_data()



Starting multi-ticker download...


/tmp/ipython-input-2344065340.py:82: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(tickers, start=start, end=end, progress=False, group_by='ticker')


Downloaded tickers. Price shape: (3986, 26)


/tmp/ipython-input-2344065340.py:128: FutureWarning: YF.download() has changed argument auto_adjust default to True
  fx = yf.download("USDINR=X", start=start, end=end, progress=False)["Close"].ffill().bfill()


Synthetic FX created (median ratio).
Final prices shape: (3986, 26)


In [ ]:
out = backtest_pipeline(prices, use_hmm=False, use_vol_spike=True)


Backtest complete. Outputs written to backtest_outputs


/tmp/ipython-input-2344065340.py:660: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annual = ann.groupby("Year").apply(lambda x: x["NAV"].iloc[-1] / x["NAV"].iloc[0] - 1)
/tmp/ipython-input-2344065340.py:668: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bench_annual = bdf.groupby("Year").apply(lambda x: x.iloc[-1,0] / x.iloc[0,0] - 1)


In [ ]:
run_reporting_and_ablation(prices)

Exported summary CSVs to backtest_outputs


/tmp/ipython-input-2344065340.py:752: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lev = leverage_series.reindex(w_norm.index).fillna(method='ffill').fillna(1.0)


Saved allocation history -> backtest_outputs/allocation_history.png


/tmp/ipython-input-2344065340.py:764: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lev = leverage_series.reindex(nav_series.index).fillna(method='ffill').fillna(1.0)


Saved leverage vs vol -> backtest_outputs/leverage_vs_vol.png
Saved monthly heatmap -> backtest_outputs/monthly_returns_heatmap.png
Saved regime-average weights -> backtest_outputs/regime_avg_weights.csv


/tmp/ipython-input-2344065340.py:660: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annual = ann.groupby("Year").apply(lambda x: x["NAV"].iloc[-1] / x["NAV"].iloc[0] - 1)
/tmp/ipython-input-2344065340.py:668: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bench_annual = bdf.groupby("Year").apply(lambda x: x.iloc[-1,0] / x.iloc[0,0] - 1)


Backtest complete. Outputs written to backtest_outputs
Ablation experiments complete. Results saved to backtest_outputs
Bootstrap test complete -> {'obs_diff': -0.032277879350719706, 'ci_low': -0.05020015962531341, 'ci_high': -0.014940735282119326, 'pval': 0.4794}
Saved NAV comparison -> backtest_outputs/nav_comparison.png
Reporting complete. Files in backtest_outputs


True